# Self-P2IR on LLR-LUS: Dice and tumour error on the same predictions

1. Self-P2IR (released weights, run as the authors' real-frame tester) on all 46 LLR-LUS images. The LUS clouds
   were made with the recipe recovered from the released P2I-LReg clouds, from hand-corrected liver masks.
   Saves per image the pose, Dice, the bent CT and the cloud, for viewing and scoring on the laptop.
2. Frame check: does Self-P2IR's accuracy change when the CT is rotated or moved in its own coordinates?
   (LLR-LUS CTs are rotated ~45 deg and placed up to 2 m away compared with P2I-LReg's.) 100 synthetic
   P2I-LReg views of patient 01, where the true pose is known.

**Before running**
1. *Add Input*: the dataset `junezhou001/p2i-lreg`, and your private LUS dataset (the uploaded `lus-selfp2ir.zip`).
2. *Settings*: **Internet on**; accelerator **GPU T4**.
3. *Save Version -> Save & Run All*. Building the renderer (PyTorch3D) takes about 15 minutes.
4. Download `selfp2ir_lus_results.zip` and unzip it into `dataset/lus-evaluation/selfp2ir_results/`.

## 1. Environment

In [ ]:
!nvidia-smi -L
!pip install -q open3d easydict nibabel iopath fvcore
import sys, torch, numpy, open3d
print(sys.version.split()[0], "torch", torch.__version__, "cuda", torch.cuda.is_available(),
      "numpy", numpy.__version__, "open3d", open3d.__version__)
assert torch.cuda.is_available(), "turn the GPU on: Settings -> Accelerator -> GPU T4"

In [ ]:
# PyTorch3D, built from source at a pinned commit. A recent PyTorch needs C++20 where its setup.py
# asks for C++17, so if the first build fails the flag is changed and the build repeated.
P3D = "/kaggle/working/src_pytorch3d"
P3D_COMMIT = "fdd657494ea5151cd66e19a1594494f1e1d592a1"
BUILD = "FORCE_CUDA=1 pip install -q --no-build-isolation --no-deps ."
try:
    import pytorch3d
except ImportError:
    !git clone -q https://github.com/facebookresearch/pytorch3d.git {P3D} && git -C {P3D} checkout -q {P3D_COMMIT}
    !cd {P3D} && ({BUILD} > /kaggle/working/p3d_build.log 2>&1 || (echo "retrying with C++20" && sed -i 's/-std=c++17/-std=c++20/g' setup.py && rm -rf build && {BUILD} 2>&1 | tail -5))
    import pytorch3d
from pytorch3d.ops import knn_points
from pytorch3d.renderer import PointsRasterizer
print("pytorch3d", pytorch3d.__version__)

## 2. Find the two datasets

In [ ]:
ORACLE = False

In [ ]:
import glob, os
hits = sorted(set(os.path.dirname(os.path.dirname(os.path.dirname(p)))
                  for depth in range(1, 7)
                  for p in glob.glob("/kaggle/input/" + "*/" * depth + "model/reconstructed_mesh_world_m.obj")
                  if os.path.exists(os.path.join(os.path.dirname(os.path.dirname(p)), "syn", "camPose.yml"))))
P2I = hits[0] if len(hits) == 1 else None
assert P2I or ORACLE, f"expected one P2I-LReg root, found {hits}"
lus = sorted(os.path.dirname(p) for p in glob.glob("/kaggle/input/**/meta.json", recursive=True)
             if os.path.exists(os.path.join(os.path.dirname(p), "01", "real", "CAM_K.yml")))
assert len(lus) == 1, f"expected one LUS upload (a folder with meta.json and 01/real/CAM_K.yml), found {lus}"
LUS_UPLOAD = lus[0]
print("P2I-LReg:", P2I)
print("LUS upload:", LUS_UPLOAD)

## 3. Self-P2IR at the pinned commit, and its pretrained weights

In [ ]:
COMMIT = "60d6c1b30d57148e6bc0c97fd1fab0f68cc0f079"
WEIGHTS_SHA256 = "e83a9110bb6a1b1bf7e49e7e0f3a3736b0b01ce9ec76626972a5406afaa963ba"
REPO = "/kaggle/working/Self-P2IR"
WEIGHTS = "/kaggle/working/model_best_loss.pth"
if not os.path.exists(REPO):
    !git clone -q https://github.com/junzastar/Self-P2IR.git {REPO}
!git -C {REPO} checkout -q {COMMIT} && git -C {REPO} rev-parse HEAD
if not os.path.exists(WEIGHTS):
    !curl -sL -o {WEIGHTS} "https://drive.usercontent.google.com/download?id=1xz1u4jtc-J3g8IpgbjHMBS9CVpCCK4nO&export=download&confirm=t"
import hashlib
digest = hashlib.sha256(open(WEIGHTS, "rb").read()).hexdigest()
assert digest == WEIGHTS_SHA256, f"weights checksum {digest} differs from the expected one"
print("weights checksum ok")

## 4. Our scripts (embedded from the repo by make_lus_notebook.py)

In [ ]:
os.makedirs('/kaggle/working/benchmark/tools', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/benchmark/scale_recipe.py
"""
The PCA scale recipe of Self-P2IR (Zhou et al., TMI 2025, Sec. III-C, Eq. 1-2), as written in the paper.

Monocular depth has no true size, so the camera cloud T is resized by
    sigma = size(S) / size(T)
where S is the CT model and size() projects the points onto their own principal axis (the
eigenvector of the covariance with the largest eigenvalue). The paper does not say how the
projections become one number; we use their range (max - min), fixed before any run.

Applied to a synthetic cloud (true size), the recipe returns the factor by which it would
resize that cloud: 1.0 means the recipe is exact for that view.

Validity check (python benchmark/scale_recipe.py): if the released real PLYs were made with
this recipe, size(CT) / size(PLY) would be exactly 1 on every frame.
"""
import os

import cv2
import numpy as np
import open3d as o3d

DATA_DIR = "dataset/liver-registration"


def principal_size(X: np.ndarray, how: str = "range") -> float:
    Xc = X - X.mean(axis=0)
    _, V = np.linalg.eigh(np.cov(Xc.T))
    proj = Xc @ V[:, -1]
    return float(np.ptp(proj) if how == "range" else proj.std())


def recipe_factor(ct_vertices: np.ndarray, cloud: np.ndarray, how: str = "range") -> float:
    """Factor the recipe applies to `cloud` (same units as `ct_vertices`)."""
    return principal_size(ct_vertices, how) / principal_size(cloud, how)


def main():
    for how in ("range", "std"):
        ratios = []
        for pid in sorted(os.listdir(DATA_DIR)):
            base = os.path.join(DATA_DIR, pid)
            if not os.path.isdir(os.path.join(base, "real", "liverPcds")):
                continue
            mesh = o3d.io.read_triangle_mesh(os.path.join(base, "model", "reconstructed_mesh_world_m.obj"))
            S = np.asarray(mesh.vertices) * 1000.0
            for fn in sorted(os.listdir(os.path.join(base, "real", "liverPcds"))):
                frame = fn[:-4]
                pts = np.asarray(o3d.io.read_point_cloud(os.path.join(base, "real", "liverPcds", fn)).points)
                mask = cv2.imread(os.path.join(base, "real", "labels", frame, "label.png")).sum(-1).reshape(-1) > 0
                ratios.append(recipe_factor(S, pts[mask], how))
        r = np.array(ratios)
        print(f"size(CT) / size(released PLY), {how:5s}: median {np.median(r):.3f}, "
              f"10-90 % {np.percentile(r, 10):.3f}-{np.percentile(r, 90):.3f}, "
              f"min {r.min():.3f}, max {r.max():.3f}  ({len(r)} frames; 1.000 everywhere = recipe reproduced)")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/benchmark/selfp2ir_predict.py
"""
Run the authors' pretrained Self-P2IR rigid network on synthetic test views; save the predicted poses.

Follows their synthetic tester (tester.py, _synTester.test_thr) step by step:
  dataset item -> collate_fn_3dmatch -> Pipeline (posenet) -> Matching.get_match(thr=0.15)
  -> MatchMotionLoss.ransac_regist_coarse (weighted SVD) -> R, t  (CT metres -> Blender camera metres)
Neighbourhood limits are calibrated as in main.py: on the augmented training set, whose first
items are patient 01's, with setup_seed(0).

Deviations from their code, all needed to run it here or to keep the test clean:
  - Only the rigid network (posenet.* weights) is built, so pytorch3d and the real-data modules
    are not needed. Weights are loaded with strict=True (their loader uses strict=False).
  - CUDA-only extensions imported by the loss module (Chamfer, EMD) and the plotting module are
    replaced by stubs; none of them is used at test time.
  - Their PoseDataset hard-codes patients 1-21; the list is replaced by --patients.
  - Their top-level folders are registered as packages explicitly, so installed packages with the
    same name (Hugging Face `datasets` on Kaggle) cannot shadow them.
  - A view their loader rejects (< 100 liver pixels or < 1000 correspondences to the TRUE pose)
    is skipped and recorded. Their test loop silently replaces it with a random other view.
  - Without a GPU, tensor.cuda() is a no-op so their solver runs on the CPU.
  - Open3D's random generator (used by their CT point sampling) is seeded, so runs repeat exactly.
Condition B resizes the target cloud after the rejection check, so A and B keep the same views.
Progress is saved every 25 views; rerunning the same command resumes, reusing the saved
neighbourhood limits (Kaggle sessions stop after 12 hours).

Usage:
  python benchmark/selfp2ir_predict.py --repo <Self-P2IR checkout> --weights model_best_loss.pth \
      --condition A [--patients 01 06 10 11] [--out benchmark/selfp2ir_out]
  --condition B             recipe scale (scale_recipe.py, range definition)
  --condition B --factor 1.11  fixed scale factor (sensitivity runs)
"""
import argparse
import hashlib
import json
import os
import sys
import time
import types

import numpy as np
import open3d as o3d
import torch
import yaml

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from scale_recipe import recipe_factor  # noqa: E402

PATIENT_LINE = "self.patients = [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21]"


def stub(name, **attrs):
    mod = types.ModuleType(name)
    mod.__dict__.update(attrs)
    sys.modules[name] = mod


def import_repo(repo):
    """Make their packages importable without the training-only CUDA extensions."""
    sys.path.insert(0, repo)

    # Their top-level folders have no __init__.py, so an installed package of the same name
    # (e.g. Hugging Face `datasets` on Kaggle) would win; register the repo folders explicitly
    for name in ("configs", "cpp_wrappers", "datasets", "extensions", "lib", "models"):
        sys.modules.pop(name, None)
        pkg = types.ModuleType(name)
        pkg.__path__ = [os.path.join(repo, name)]
        sys.modules[name] = pkg

    class TrainingOnly(torch.nn.Module):  # lib/loss.py builds one at import time
        def forward(self, *args, **kwargs):
            raise RuntimeError("training-only CUDA extension, not available here")

    stub("extensions.chamfer_distance.chamfer_distance", ChamferDistance=TrainingOnly)
    stub("extensions.earth_movers_distance.emd", EarthMoverDistance=TrainingOnly)
    stub("lib.vis")  # mayavi / skimage plotting, only called in debug branches
    try:
        import torchvision.transforms  # noqa: F401
    except ImportError:
        stub("torchvision")
        stub("torchvision.transforms")
    try:
        import scipy.misc  # noqa: F401
    except ImportError:
        stub("scipy.misc")

    # Their dataset hard-codes patients 1-21; read the list from a module variable instead
    path = os.path.join(repo, "datasets", "dataset.py")
    src = open(path).read()
    assert src.count(PATIENT_LINE) == 1, "dataset.py changed: patient list not found"
    mod = types.ModuleType("datasets.dataset")
    mod.__file__ = path
    mod.PATIENTS = None
    sys.modules["datasets.dataset"] = mod
    exec(compile(src.replace(PATIENT_LINE, "self.patients = PATIENTS"), path, "exec"), mod.__dict__)
    return mod


def load_config(repo, data_root, device):
    from easydict import EasyDict as edict
    from configs.models import architectures

    yaml.add_constructor("!join", lambda loader, node: "_".join(str(i) for i in loader.construct_sequence(node)),
                         Loader=yaml.Loader)
    with open(os.path.join(repo, "configs", "test", "main_config.yaml")) as f:
        config = edict(yaml.load(f, Loader=yaml.Loader))
    config.dataset = "syn"
    config.data_root = os.path.abspath(data_root)
    config.device = device
    config.kpfcn_config.architecture = architectures["syn"]
    return config


def build_model(config, weights):
    from models.lepard.lepard import Pipeline

    state = torch.load(weights, map_location="cpu", weights_only=False)["state_dict"]
    posenet = {k[len("posenet."):]: v for k, v in state.items() if k.startswith("posenet.")}
    model = Pipeline(config)
    model.load_state_dict(posenet, strict=True)
    return model.to(config.device).eval()


def to_device(batch, device):
    """Same rule as their tester loop."""
    for k, v in batch.items():
        if isinstance(v, list):
            if v and not isinstance(v[0], (str, np.ndarray)):
                batch[k] = [item.to(device) for item in v]
        elif not isinstance(v, (dict, float, type(None), np.ndarray)):
            batch[k] = v.to(device)
    return batch


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--repo", required=True)
    ap.add_argument("--weights", required=True)
    ap.add_argument("--data-root", default="dataset/liver-registration")
    ap.add_argument("--patients", nargs="+", default=["01", "06", "10", "11"])
    ap.add_argument("--condition", choices=["A", "B"], required=True)
    ap.add_argument("--factor", type=float, default=None, help="fixed scale for B (default: recipe)")
    ap.add_argument("--limit", type=int, default=None, help="first N test views only (smoke test)")
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--ct-rot-z", type=float, default=0.0,
                    help="rotate the CT about its z axis by this many degrees before registration (frame test)")
    ap.add_argument("--ct-shift", type=float, nargs=3, default=(0.0, 0.0, 0.0),
                    help="then shift the CT by this vector, metres (frame test)")
    ap.add_argument("--out", default="benchmark/selfp2ir_out")
    args = ap.parse_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type == "cpu":
        torch.Tensor.cuda = lambda self, *a, **k: self  # their solver calls .cuda() directly

    ds_mod = import_repo(os.path.abspath(args.repo))
    from datasets.dataloader import calibrate_neighbors, collate_fn_3dmatch
    from lib.logger import setup_seed
    from lib.loss import MatchMotionLoss as MML
    from models.lepard.matching import Matching as CM

    config = load_config(args.repo, args.data_root, device)
    patients = sorted(int(p) for p in args.patients)
    o3d.utility.random.seed(args.seed)

    tag = "A" if args.condition == "A" else ("B_recipe" if args.factor is None else f"B_x{args.factor:.2f}")
    a = np.deg2rad(args.ct_rot_z)
    G = np.array([[np.cos(a), -np.sin(a), 0], [np.sin(a), np.cos(a), 0], [0, 0, 1]], np.float32)
    d = np.array(args.ct_shift, np.float32)
    if args.ct_rot_z or d.any():
        tag += f"_ctrot{args.ct_rot_z:g}_ctshift{'_'.join(f'{v:g}' for v in d)}"
    os.makedirs(args.out, exist_ok=True)
    out_path = os.path.join(args.out, f"predictions_{tag}.json")
    records, rejected, limits = [], [], None
    if os.path.exists(out_path):
        saved = json.load(open(out_path))
        if saved["meta"].get("complete"):
            print(f"{out_path} is complete; delete it to rerun")
            return
        records, rejected = saved["predictions"], [tuple(x) for x in saved["meta"]["rejected"]]
        limits = np.array(saved["meta"]["neighbourhood_limits"])
        print(f"resuming {out_path}: {len(records)} done, {len(rejected)} rejected")
    done = {(r["patient"], r["view"]) for r in records} | set(rejected)

    # Neighbourhood limits, as in main.py: augmented training set, patient 01's views first
    if limits is None:
        setup_seed(0)
        ds_mod.PATIENTS = [patients[0]]
        train_set = ds_mod.PoseDataset("train", config, data_augmentation=True)
        limits = calibrate_neighbors(train_set, config.kpfcn_config, collate_fn=collate_fn_3dmatch)
    print("neighbourhood limits:", limits.tolist())

    model = build_model(config, args.weights)
    ds_mod.PATIENTS = patients
    test_set = ds_mod.PoseDataset("test", config, data_augmentation=False)
    ct_vertices = {p: np.asarray(o3d.io.read_triangle_mesh(
        os.path.join(config.data_root, f"{p:02d}", "model", "reconstructed_mesh_world_m.obj")).vertices)
        for p in patients}

    meta = dict(
        condition=tag, scale_definition="range" if args.factor is None else "fixed",
        repo=os.path.abspath(args.repo), weights_sha256=sha256(args.weights),
        device=str(device), torch=torch.__version__, neighbourhood_limits=limits.tolist(),
        patients=patients, frames="R, t map CT points (metres) to the Blender camera (metres, -Z forward); "
                                  "R_true, t_true are the same from camPose.yml")

    def save(complete):
        with open(out_path, "w") as f:
            json.dump(dict(meta={**meta, "complete": complete, "n_views": len(records), "rejected": rejected},
                           predictions=records), f)

    n = len(test_set) if args.limit is None else min(args.limit, len(test_set))
    t0 = time.time()
    for i in range(n):
        patient, view = int(test_set.list_patient[i]), int(test_set.list_rank[i])
        if (patient, view) in done:
            continue
        np.random.seed(args.seed + i)  # their item sampling uses the global numpy RNG
        item = test_set.get_item(i)
        if item is None:
            rejected.append((patient, view))
            continue

        if args.ct_rot_z or d.any():  # move the CT; the true pose follows so that it still maps CT -> camera
            item["preope_pcd_src"] = item["preope_pcd_src"] @ G.T + d
            item["pose_R"] = item["pose_R"] @ G.T
            item["pose_t"] = item["pose_t"] - item["pose_R"] @ d.reshape(3, 1)
        scale = 1.0
        if args.condition == "B":
            if args.factor is None:
                raw = np.asarray(o3d.io.read_point_cloud(test_set.list_liverPcd[i]).points) / 1000.0
                scale = recipe_factor(ct_vertices[patient], raw)
            else:
                scale = args.factor
            item["intra_liver_pcd_tgt"] = item["intra_liver_pcd_tgt"] * np.float32(scale)

        batch = to_device(collate_fn_3dmatch([item], config.kpfcn_config, neighborhood_limits=limits), device)
        with torch.no_grad():
            data = model(batch)
            match_pred, _, _ = CM.get_match(data["conf_matrix_pred"], thr=0.15, mutual=False)
            rot, trn, _, _ = MML.ransac_regist_coarse(data["s_pcd"], data["t_pcd"], data["src_mask"],
                                                     data["tgt_mask"], match_pred, data)

        records.append(dict(
            patient=patient, view=view, scale=scale, n_matches=int(len(match_pred)),
            R=np.asarray(rot[0].detach().cpu(), float).tolist(),
            t=np.asarray(trn[0].detach().cpu(), float).reshape(3).tolist(),
            R_true=np.asarray(item["pose_R"], float).tolist(),
            t_true=np.asarray(item["pose_t"], float).reshape(3).tolist()))
        if len(records) % 25 == 0:
            save(complete=False)
            print(f"[{tag}] {i + 1}/{n} views, {len(rejected)} rejected, {time.time() - t0:.0f} s")

    save(complete=True)
    print(f"saved {len(records)} predictions ({len(rejected)} rejected) to {out_path}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/benchmark/selfp2ir_real.py
"""
Run the authors' pretrained Self-P2IR on P2I-LReg real test frames; save their Dice and Chamfer per frame.

Purpose: check that the released weights reproduce the paper before the model is used elsewhere.
Paper (Zhou et al., TMI 2025, Table II, mean of five folds): Dice 78.89 +- 6.76 %, CD 2.97 +- 1.06.
The released weights are one fold, so the result should fall near that range, not on it.

Follows their real tester (tester.py, _realTester.test_thr) step by step:
  dataset item -> collate_fn_3dmatch -> DeformRegis: rigid network, then the deformation pyramid
  optimised on this frame, then the bent CT points drawn into the image -> Dice with the liver label,
  CD between the frame's cloud and the bent CT points.
Neighbourhood limits are calibrated as in main.py, on the real training set with setup_seed(0).

Deviations from their code, all needed to run it on a current PyTorch:
  - Their PointNet++ CUDA extension is replaced by tools/pointnet2_torch.py (same operations in
    plain PyTorch) and their Chamfer CUDA extension by `ChamferDistance` below (same numbers:
    squared nearest distances in both directions).
  - Plotting and training-only modules are replaced by stubs, as in selfp2ir_predict.py.
  - Weights are loaded with strict=True for every tensor the file holds (their loader: strict=False).
  - A frame their loader rejects (< 100 liver pixels) is skipped and recorded.
  - The rendered depth is not turned into points: that output is used in training only, and their
    helper (lib/geometry.depth2pointcloud) is GPU-only and fails when the drawn liver is small.
  - Without a GPU, tensor.cuda() is a no-op.

Besides their two numbers, two plain distances are saved per frame, in mm (not squared):
  cloud_to_ct_mm  mean distance from the frame's cloud to the nearest bent CT point
  ct_to_cloud_mm  mean distance from every CT point (the whole liver) to the nearest cloud point
Their "CD" is mean(squared metres, both directions added) x 1000.

Usage:
  python studies/benchmark/selfp2ir_real.py --repo <Self-P2IR checkout> --weights model_best_loss.pth \
      [--data-root dataset/liver-registration] [--patients 01 02 ...] [--limit N] [--out studies/benchmark/selfp2ir_real_out]
"""
import argparse
import importlib
import importlib.util
import json
import os
import sys
import time

import numpy as np
import open3d as o3d
import torch

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from selfp2ir_predict import import_repo, load_config, sha256, stub, to_device  # noqa: E402

HERE = os.path.dirname(os.path.abspath(__file__))


class ChamferDistance(torch.nn.Module):
    """Squared distance from each point to its nearest point in the other set, both ways (as chamfer3D.cu)."""

    def forward(self, a, b):
        d = torch.cdist(a, b) ** 2
        return d.min(dim=2).values, d.min(dim=1).values


def optional(name, **attrs):
    try:
        importlib.import_module(name)
    except ImportError:
        stub(name, **attrs)


def import_real(repo, device):
    """Their real-frame modules, importable on a current PyTorch."""
    ds_mod = import_repo(repo)
    stub("extensions.chamfer_distance.chamfer_distance", ChamferDistance=ChamferDistance)
    stub("lib.vis", project_mesh_to_image=None, visualize_pcds=None)
    for name, attrs in [("skimage", dict(io=None, img_as_ubyte=None)), ("imageio", {}), ("h5py", {}),
                        ("geomloss", dict(SamplesLoss=None)), ("tqdm.notebook", dict(tqdm=None))]:
        optional(name, **attrs)
    spec = importlib.util.spec_from_file_location("models.pointnet.pointnet2.pointnet2_utils",
                                                  os.path.join(HERE, "tools", "pointnet2_torch.py"))
    ops = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(ops)
    sys.modules[spec.name] = ops

    import models.mynetwork as net
    net.depth2pointcloud = lambda depth, K, downsampling, sampling_points: \
        torch.zeros(depth.shape[0], sampling_points, 3, device=depth.device)
    if device.type == "cpu":
        torch.Tensor.cuda = lambda self, *a, **k: self
    return ds_mod, net


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--repo", required=True)
    ap.add_argument("--weights", required=True)
    ap.add_argument("--data-root", default="dataset/liver-registration")
    ap.add_argument("--patients", nargs="+", default=None, help="default: every patient in the data root")
    ap.add_argument("--limit", type=int, default=None, help="first N test frames only (smoke test)")
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--save-points", default=None,
                    help="folder: per frame an .npz with R, t, the bent CT points and the cloud points (their frame: "
                         "Blender camera axes, metres, after the cloud's shift) and the rendered mask, for viewing")
    ap.add_argument("--limits", type=int, nargs=4, default=None,
                    help="neighbourhood limits to use instead of calibrating them on the data root's training frames "
                         "(LLR-LUS has none: use the P2I-LReg calibration, [9, 19, 25, 30] on all 21 patients)")
    ap.add_argument("--out", default=os.path.join(HERE, "selfp2ir_real_out"))
    args = ap.parse_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ds_mod, net = import_real(os.path.abspath(args.repo), device)
    from configs.models import architectures
    from datasets.dataloader import calibrate_neighbors, collate_fn_3dmatch
    from lib.logger import setup_seed
    from lib.loss import BinaryDiceLoss

    config = load_config(args.repo, args.data_root, device)
    config.dataset = "real"
    config.kpfcn_config.architecture = architectures["real"]
    patients = sorted(int(p) for p in (args.patients or
                                       [d for d in os.listdir(config.data_root) if d.isdigit()]))
    o3d.utility.random.seed(args.seed)

    setup_seed(0)
    ds_mod.PATIENTS = patients
    if args.limits:
        limits = np.array(args.limits)
    else:
        train_set = ds_mod.PoseDataset("train", config, data_augmentation=True)
        limits = calibrate_neighbors(train_set, config.kpfcn_config, collate_fn=collate_fn_3dmatch)
    print("neighbourhood limits:", limits.tolist())

    model = net.DeformRegis(config)
    state = torch.load(args.weights, map_location="cpu", weights_only=False)["state_dict"]
    missing, unexpected = model.load_state_dict(state, strict=False)
    assert not unexpected and not missing, f"weights do not match the model: missing {missing}, unexpected {unexpected}"
    model = model.to(device).eval()
    for name, param in model.named_parameters():  # as their tester
        param.requires_grad = "deformNet" in name

    test_set = ds_mod.PoseDataset("test", config, data_augmentation=False)
    dice_loss, chamfer = BinaryDiceLoss(), ChamferDistance()
    records, rejected = [], []
    n = len(test_set) if args.limit is None else min(args.limit, len(test_set))
    os.makedirs(args.out, exist_ok=True)
    out_path = os.path.join(args.out, "real_frames.json")
    t0 = time.time()
    for i in range(n):
        patient = int(test_set.list_patient[i])
        frame = os.path.basename(os.path.dirname(test_set.list_label[i]))
        np.random.seed(args.seed + i)  # their item sampling uses the global numpy RNG
        item = test_set.get_item(i)
        if item is None:
            rejected.append((patient, frame))
            continue
        batch = to_device(collate_fn_3dmatch([item], config.kpfcn_config, neighborhood_limits=limits), device)
        data = model(batch)
        with torch.no_grad():
            R, t = data["R_s2t_pred"], data["t_s2t_pred"]
            bent = torch.bmm(R, data["deformed_src"].permute(0, 2, 1)).permute(0, 2, 1) + t.permute(0, 2, 1)
            d_cloud, d_ct = chamfer(data["batched_tgt_pcd"], bent)
            records.append(dict(
                patient=patient, frame=frame, scale=float(item["scale"]),
                dice=float(100.0 * (1.0 - dice_loss(data["mask"], data["liver_label"]))),
                cd=float(((d_cloud + d_ct) * 1000.0).mean()),
                cloud_to_ct_mm=float(d_cloud.sqrt().mean() * 1000.0),
                ct_to_cloud_mm=float(d_ct.sqrt().mean() * 1000.0),
                R=R[0].cpu().numpy().astype(float).tolist(), t=t[0].cpu().numpy().astype(float).reshape(3).tolist()))
            if args.save_points:
                os.makedirs(args.save_points, exist_ok=True)
                rendered = (data["mask"][0] > 0).cpu().numpy()
                np.savez_compressed(os.path.join(args.save_points, f"{patient:02d}_{frame}.npz"),
                                    R=R[0].cpu().numpy(), t=t[0].cpu().numpy(), bent=bent[0].cpu().numpy(),
                                    cloud=data["batched_tgt_pcd"][0].cpu().numpy(),
                                    rendered_mask=np.packbits(rendered), mask_shape=np.array(rendered.shape))
        r = records[-1]
        print(f"patient {patient:02d} {frame}: dice {r['dice']:.1f}  cd {r['cd']:.2f}  "
              f"cloud->ct {r['cloud_to_ct_mm']:.1f} mm  ct->cloud {r['ct_to_cloud_mm']:.1f} mm  "
              f"({time.time() - t0:.0f} s)", flush=True)
        with open(out_path, "w") as f:
            json.dump(dict(meta=dict(repo=os.path.abspath(args.repo), weights_sha256=sha256(args.weights),
                                     device=str(device), torch=torch.__version__, patients=patients,
                                     neighbourhood_limits=limits.tolist(), rejected=rejected,
                                     complete=i == n - 1), frames=records), f)

    dice, cd = np.array([r["dice"] for r in records]), np.array([r["cd"] for r in records])
    print(f"\n{len(records)} frames from patients {sorted({r['patient'] for r in records})}, {len(rejected)} rejected")
    print(f"Dice {dice.mean():.2f} +- {dice.std():.2f} %   (paper, five folds: 78.89 +- 6.76)")
    print(f"CD   {cd.mean():.2f} +- {cd.std():.2f}     (paper, five folds: 2.97 +- 1.06)")
    for key in ("cloud_to_ct_mm", "ct_to_cloud_mm"):
        print(f"{key}: mean {np.mean([r[key] for r in records]):.1f} mm")
    print("saved", out_path)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/benchmark/lus_kaggle_pack.py
"""
Pack the LLR-LUS Self-P2IR layout (lus_selfp2ir_prep.py) small enough to upload to Kaggle, and unpack it there.

The clouds are most of the size (4 GB) but carry nothing beyond their depth map: each released-style cloud is
    P = sigma * [(u - W/2) D / fx, (v - H/2) D / fy, D] - shift      (every pixel, D = 0 off the liver)
So `pack` stores D (float32, compressed) instead of the cloud, plus a fingerprint of the cloud, and `unpack`
rebuilds every cloud and checks it against its fingerprint.

  pack     dataset/lus-evaluation/selfp2ir/ -> dataset/lus-evaluation/kaggle_upload/lus-selfp2ir.zip
  unpack   <unzipped upload> -> <out>/ (the layout Self-P2IR's loader reads)

LLR-LUS is research-only: upload it only as a PRIVATE Kaggle dataset, with the authors' agreement.

  centre-ct  <layout> -> <out>/ : the same layout with each CT liver moved to the origin (see centre_ct)

Usage: python studies/benchmark/lus_kaggle_pack.py pack
       python lus_kaggle_pack.py unpack <unzipped folder> <out folder> [<sigma json, e.g. oracle.json>]
       python lus_kaggle_pack.py centre-ct <layout> <out folder>
"""
import json
import os
import shutil
import sys
import zipfile

import numpy as np
import open3d as o3d

SRC = "dataset/lus-evaluation/selfp2ir"
UPLOAD = "dataset/lus-evaluation/kaggle_upload"


def rebuild(D, sigma, shift, fx, fy):
    H, W = D.shape
    v, u = np.mgrid[0:H, 0:W]
    T = np.stack([(u - W / 2) * D / fx, (v - H / 2) * D / fy, D], -1).reshape(-1, 3)
    return sigma * T - np.asarray(shift)


def fingerprint(P):
    return [float(x) for x in np.r_[np.abs(P).mean(0), np.abs(P).max(0)]]  # mm; the cloud is centred, so not sums


def pack():
    meta = json.load(open(f"{SRC}/meta.json"))
    stage = f"{UPLOAD}/lus-selfp2ir"
    shutil.rmtree(stage, ignore_errors=True)
    shutil.copytree(SRC, stage, ignore=shutil.ignore_patterns("liverPcds"))
    for key, m in meta.items():
        P = np.asarray(o3d.io.read_point_cloud(f"{SRC}/{m['id']}/real/liverPcds/{m['frame']}.ply").points)
        W, H = m["size"]
        D = ((P + np.asarray(m["shift_mm"]))[:, 2] / m["sigma"]).reshape(H, W).astype(np.float32)
        err = np.abs(rebuild(D, m["sigma"], m["shift_mm"], m["fx"], m["fy"]) - P).max()
        assert err < 1e-3, f"{key}: depth does not rebuild the cloud (max error {err} mm)"
        os.makedirs(f"{stage}/{m['id']}/real/depth", exist_ok=True)
        np.savez_compressed(f"{stage}/{m['id']}/real/depth/{m['frame']}.npz", D=D)
        m["fingerprint"] = fingerprint(P)
        print(f"{key}: packed, rebuild error {err:.1e} mm", flush=True)
    json.dump(meta, open(f"{stage}/meta.json", "w"), indent=1)
    zpath = f"{UPLOAD}/lus-selfp2ir.zip"
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
        for root, _, files in os.walk(stage):
            for f in files:
                z.write(os.path.join(root, f), os.path.relpath(os.path.join(root, f), UPLOAD))
    shutil.rmtree(stage)
    print(f"wrote {zpath} ({os.path.getsize(zpath) / 1e6:.0f} MB)")


def unpack(src, out, sigma_json=None):
    """Rebuild the clouds. With sigma_json ({image: {"sigma": ...}}, e.g. lus_oracle.py's oracle.json) each cloud
    is then rescaled to that sigma, about the same centre, and meta.json / scale files are updated to match."""
    meta = json.load(open(f"{src}/meta.json"))
    sigmas = json.load(open(sigma_json)) if sigma_json else {}
    shutil.copytree(src, out, ignore=shutil.ignore_patterns("depth"), dirs_exist_ok=True)
    for key, m in meta.items():
        D = np.load(f"{src}/{m['id']}/real/depth/{m['frame']}.npz")["D"].astype(np.float64)
        P = rebuild(D, m["sigma"], m["shift_mm"], m["fx"], m["fy"])
        fp = np.array(fingerprint(P)); ref = np.array(m["fingerprint"])
        assert np.allclose(fp, ref, rtol=0, atol=1e-3), f"{key}: rebuilt cloud differs from the original"
        if key in sigmas:
            centre = np.asarray(m["shift_mm"]) / (m["sigma"] - 1)       # the shift is (sigma - 1) * centre
            m.update(sigma_recipe=m["sigma"], sigma=sigmas[key]["sigma"])
            m["shift_mm"] = ((m["sigma"] - 1) * centre).tolist()
            P = rebuild(D, m["sigma"], m["shift_mm"], m["fx"], m["fy"])
            with open(f"{out}/{m['id']}/real/scale/{m['frame']}.txt", "w") as f:
                f.write(f"{m['sigma']}\n")
        os.makedirs(f"{out}/{m['id']}/real/liverPcds", exist_ok=True)
        o3d.io.write_point_cloud(f"{out}/{m['id']}/real/liverPcds/{m['frame']}.ply",
                                 o3d.geometry.PointCloud(o3d.utility.Vector3dVector(P)), write_ascii=False)
    json.dump(meta, open(f"{out}/meta.json", "w"), indent=1)
    print(f"rebuilt {len(meta)} clouds in {out}/, all match their fingerprints"
          + (f"; {len(sigmas)} rescaled to the given sigma" if sigmas else ""))


def centre_ct(src, out):
    """A copy of the layout whose CT livers are moved so their vertex mean is at the origin, as P2I-LReg's
    are (within ~3 cm). Self-P2IR is not indifferent to where the CT sits (frame check, 2026-11-13); the
    clouds are untouched (sigma ignores position). Writes <out>/ct_offsets.json: the mean subtracted, mm."""
    offsets = {}
    for pid in sorted(d for d in os.listdir(src) if d.isdigit()):
        os.makedirs(f"{out}/{pid}/model", exist_ok=True)
        for name in os.listdir(f"{src}/{pid}"):
            if name != "model" and not os.path.exists(f"{out}/{pid}/{name}"):
                os.symlink(os.path.abspath(f"{src}/{pid}/{name}"), f"{out}/{pid}/{name}")
        mesh = o3d.io.read_triangle_mesh(f"{src}/{pid}/model/reconstructed_mesh_world_m.obj")
        V = np.asarray(mesh.vertices)
        offsets[pid] = (V.mean(0) * 1000.0).tolist()
        mesh.vertices = o3d.utility.Vector3dVector(V - V.mean(0))
        o3d.io.write_triangle_mesh(f"{out}/{pid}/model/reconstructed_mesh_world_m.obj", mesh)
    json.dump(offsets, open(f"{out}/ct_offsets.json", "w"), indent=1)
    print("CT livers moved to the origin; offsets (mm):", {k: np.round(v).astype(int).tolist() for k, v in offsets.items()})


if __name__ == "__main__":
    if sys.argv[1:2] == ["pack"]:
        pack()
    elif sys.argv[1:2] == ["unpack"] and len(sys.argv) in (4, 5):
        unpack(sys.argv[2], sys.argv[3], sys.argv[4] if len(sys.argv) == 5 else None)
    elif sys.argv[1:2] == ["centre-ct"] and len(sys.argv) == 4:
        centre_ct(sys.argv[2], sys.argv[3])
    else:
        print(__doc__)

In [ ]:
%%writefile /kaggle/working/benchmark/tools/build_cpp_wrappers.py
"""
Build Self-P2IR's two KPConv C++ helpers (grid subsampling, radius neighbours) for this machine.

Their setup.py needs numpy.distutils (removed from recent NumPy/setuptools), and the wrapper glue
uses NumPy 1 C-API spellings. We copy cpp_wrappers to a build folder, add a small compatibility
shim to the two wrapper.cpp files (no change to the neighbour or subsampling algorithms), build
there, and copy the .so files next to the original sources so `import cpp_wrappers...` works.

Usage: python build_cpp_wrappers.py <Self-P2IR checkout> <build folder>
"""
import glob
import os
import shutil
import sys

import numpy as np
from setuptools import Extension, setup

SHIM = """
// --- NumPy 1/2 compatibility shim (added for this build only) ---
#ifndef NPY_IN_ARRAY
#define NPY_IN_ARRAY NPY_ARRAY_IN_ARRAY
#endif
#define PyArray_NDIM(o) PyArray_NDIM((PyArrayObject*)(o))
#define PyArray_DIM(o, i) PyArray_DIM((PyArrayObject*)(o), i)
#define PyArray_DATA(o) PyArray_DATA((PyArrayObject*)(o))
// -----------------------------------------------------------------
"""

MODULES = {
    "cpp_subsampling": ("grid_subsampling", ["grid_subsampling/grid_subsampling.cpp", "wrapper.cpp"]),
    "cpp_neighbors": ("radius_neighbors", ["neighbors/neighbors.cpp", "wrapper.cpp"]),
}


def main():
    repo, build = os.path.abspath(sys.argv[1]), os.path.abspath(sys.argv[2])
    src = os.path.join(repo, "cpp_wrappers")
    dst = os.path.join(build, "cpp_wrappers")
    shutil.rmtree(dst, ignore_errors=True)
    shutil.copytree(src, dst, ignore=shutil.ignore_patterns("*.so"))

    for folder, (name, sources) in MODULES.items():
        base = os.path.join(dst, folder)
        wrapper = os.path.join(base, "wrapper.cpp")
        text = open(wrapper).read()
        anchor = "#include <numpy/arrayobject.h>"
        assert anchor in text, f"{wrapper}: numpy include not found"
        open(wrapper, "w").write(text.replace(anchor, anchor + SHIM, 1))

        os.chdir(base)
        setup(name=name, script_args=["build_ext", "--inplace"],
              ext_modules=[Extension(name, sources=["../cpp_utils/cloud/cloud.cpp"] + sources,
                                     include_dirs=[np.get_include()], extra_compile_args=["-std=c++11"])])
        for so in glob.glob(os.path.join(base, f"{name}*.so")):
            shutil.copy2(so, os.path.join(src, folder))
            print("built", os.path.join(src, folder, os.path.basename(so)))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/benchmark/tools/pointnet2_torch.py
"""
Self-P2IR's PointNet++ operations in plain PyTorch, so the real-frame network runs without compiling
their CUDA extension (pointnet2_cuda, written for PyTorch 1.10).

Drop-in for models/pointnet/pointnet2/pointnet2_utils.py. Each function follows the CUDA kernel in
models/pointnet/pointnet2/src/ line by line:
  furthest_point_sample  starts at point 0; each next point is the one farthest from those chosen
  ball_query             the first `nsample` points (in index order) closer than `radius`; a ball with
                         fewer points is filled with its first point; an empty ball is all zeros
  three_nn               the 3 nearest points and their distances (not squared)
gather / grouping / three_interpolate are index look-ups.
"""
import torch
import torch.nn as nn


def furthest_point_sample(xyz, npoint):
    """xyz (B, N, 3) -> indices (B, npoint)."""
    B, N, _ = xyz.shape
    idx = torch.zeros(B, npoint, dtype=torch.long, device=xyz.device)
    dist = torch.full((B, N), 1e10, device=xyz.device)
    last = torch.zeros(B, dtype=torch.long, device=xyz.device)
    rows = torch.arange(B, device=xyz.device)
    for j in range(1, npoint):
        dist = torch.minimum(dist, ((xyz - xyz[rows, last][:, None]) ** 2).sum(-1))
        last = dist.argmax(dim=1)
        idx[:, j] = last
    return idx


def gather_operation(features, idx):
    """features (B, C, N), idx (B, npoint) -> (B, C, npoint)."""
    return torch.gather(features, 2, idx.long()[:, None].expand(-1, features.shape[1], -1))


def grouping_operation(features, idx):
    """features (B, C, N), idx (B, npoint, nsample) -> (B, C, npoint, nsample)."""
    B, npoint, nsample = idx.shape
    return gather_operation(features, idx.reshape(B, -1)).reshape(B, -1, npoint, nsample)


def ball_query(radius, nsample, xyz, new_xyz):
    """xyz (B, N, 3), centres new_xyz (B, npoint, 3) -> indices (B, npoint, nsample)."""
    N = xyz.shape[1]
    inside = torch.cdist(new_xyz, xyz) ** 2 < radius * radius
    order = torch.where(inside, torch.arange(N, device=xyz.device), N).sort(dim=2).values[:, :, :nsample]
    first = order[:, :, :1]
    idx = torch.where(order < N, order, first)       # short ball: repeat its first point
    return torch.where(first < N, idx, 0)            # empty ball: zeros


def three_nn(unknown, known):
    """unknown (B, n, 3), known (B, m, 3) -> distances (B, n, 3), indices (B, n, 3)."""
    d2, idx = (torch.cdist(unknown, known) ** 2).topk(3, dim=2, largest=False)
    return torch.sqrt(d2), idx


def three_interpolate(features, idx, weight):
    """features (B, C, m), idx and weight (B, n, 3) -> (B, C, n)."""
    return (grouping_operation(features, idx) * weight[:, None]).sum(-1)


class QueryAndGroup(nn.Module):
    def __init__(self, radius, nsample, use_xyz=True):
        super().__init__()
        self.radius, self.nsample, self.use_xyz = radius, nsample, use_xyz

    def forward(self, xyz, new_xyz, features=None):
        idx = ball_query(self.radius, self.nsample, xyz, new_xyz)
        grouped = grouping_operation(xyz.transpose(1, 2).contiguous(), idx) - new_xyz.transpose(1, 2).unsqueeze(-1)
        if features is None:
            return grouped
        grouped_features = grouping_operation(features, idx)
        return torch.cat([grouped, grouped_features], dim=1) if self.use_xyz else grouped_features


class GroupAll(nn.Module):
    def __init__(self, use_xyz=True):
        super().__init__()
        self.use_xyz = use_xyz

    def forward(self, xyz, new_xyz, features=None):
        grouped = xyz.transpose(1, 2).unsqueeze(2)
        if features is None:
            return grouped
        return torch.cat([grouped, features.unsqueeze(2)], dim=1) if self.use_xyz else features.unsqueeze(2)

## 5. Build the two C++ helpers, and rebuild the LUS clouds from their depth maps

In [ ]:
%cd /kaggle/working
!python benchmark/tools/build_cpp_wrappers.py {REPO} /kaggle/working/build 2>&1 | grep -E "^built| error"
LUS = "/kaggle/working/lus_selfp2ir"
!python -W ignore benchmark/lus_kaggle_pack.py unpack {LUS_UPLOAD} {LUS} {"/kaggle/working/oracle.json" if ORACLE else ""}

## 6. Self-P2IR on the 46 LUS images, twice
LUS has no training frames, so the neighbourhood limits are the ones calibrated on P2I-LReg's real training
frames (all 21 patients): 9 19 25 30. One line per image.

- **as given**: the LUS CT livers in their own coordinates (rotated ~45 deg and up to 2 m from the origin);
- **centred**: the same CTs moved so their mean is at the origin, as P2I-LReg's are. The first frame check
  showed Self-P2IR loses accuracy when the CT is moved, so this is the fair placement. Clouds are identical.

In [ ]:
# Sanity check that this is the corrected upload (2026-11-13): the cloud must be scaled about its bounding-box
# centre, which leaves the PLY's bounding-box centre off zero (the first, wrong upload centred it exactly).
import numpy as np
P = np.asarray(open3d.io.read_point_cloud(f"{LUS}/01/real/liverPcds/frame_2_json.ply").points)
c = (P.max(0) + P.min(0)) / 2
print("bounding-box centre of 01/frame_2 (mm):", np.round(c, 3))
assert np.abs(c).max() > 0.05, "this is the OLD upload: upload the corrected lus-selfp2ir.zip as a new dataset version"

In [ ]:
if ORACLE:
    RUNS_LUS = {"lus_oracle": LUS}
else:
    RUNS_LUS = {"lus_results": LUS, "lus_results_centred": "/kaggle/working/lus_selfp2ir_centred"}
    !python -W ignore benchmark/lus_kaggle_pack.py centre-ct {LUS} {RUNS_LUS["lus_results_centred"]}
for out, root in RUNS_LUS.items():
    print("=====", out)
    OUT = f"/kaggle/working/{out}"
    !python -W ignore benchmark/selfp2ir_real.py --repo {REPO} --weights {WEIGHTS} --data-root {root} --patients 01 02 04 05 --limits 9 19 25 30 --out {OUT} --save-points {OUT}/points 2>&1 | grep -v "buffer loaded"
    !cp {root}/meta.json {OUT}/
    if os.path.exists(f"{root}/ct_offsets.json"):
        !cp {root}/ct_offsets.json {OUT}/

## 7. Frame check: does moving the CT change Self-P2IR's answer?
Already run once (2026-11-13); set RUN_FRAME_CHECK = True to repeat it.

In [ ]:
RUN_FRAME_CHECK = False

### 7a. The check
Same 100 synthetic views of P2I-LReg patient 01, four times: CT as given; rotated 45 deg about z; moved 0.25 m;
moved 2 m (LUS Patient 3's CT sits ~2 m from the origin). Registration recall at 5 mm, as in the paper.

In [ ]:
FC = "/kaggle/working/frame_check"
os.makedirs(FC, exist_ok=True)
RUNS = {"as given": "", "rotated 45 deg": "--ct-rot-z 45", "moved 0.25 m": "--ct-shift 0 0 0.25", "moved 2 m": "--ct-shift 0 0 2.0"}
for name, extra in (RUNS.items() if RUN_FRAME_CHECK else []):
    print("=====", name)
    !python -W ignore benchmark/selfp2ir_predict.py --repo {REPO} --weights {WEIGHTS} --data-root {P2I} --condition A --patients 01 --limit 100 {extra} --out {FC} 2>&1 | grep -E "saved|rror"

In [ ]:
import json, re, numpy as np
V = np.asarray(open3d.io.read_triangle_mesh(f"{P2I}/01/model/reconstructed_mesh_world_m.obj").vertices)  # metres
print(f"{'CT placement':38s} views  recall<5mm  median error")
for f in sorted(glob.glob(f"{FC}/predictions_*.json")):
    tag = os.path.basename(f)[len("predictions_"):-5]
    rot = float(re.search(r"ctrot([-\d.]+)", tag).group(1)) if "ctrot" in tag else 0.0
    shift = np.array(re.search(r"ctshift(.+)$", tag).group(1).split("_"), float) if "ctshift" in tag else np.zeros(3)
    a = np.deg2rad(rot)
    G = np.array([[np.cos(a), -np.sin(a), 0], [np.sin(a), np.cos(a), 0], [0, 0, 1]])
    X = V @ G.T + shift                                    # the CT as the model saw it
    errs = []
    for r in json.load(open(f))["predictions"]:
        R, t, R0, t0 = (np.array(r[k]) for k in ("R", "t", "R_true", "t_true"))
        errs.append(1000 * np.linalg.norm(X @ (R - R0).T + (t - t0), axis=1).mean())
    errs = np.array(errs)
    print(f"{tag:38s} {len(errs):5d}  {100 * (errs < 5).mean():9.1f} %  {np.median(errs):9.2f} mm")

## 8. Download

In [ ]:
!cd /kaggle/working && zip -qr selfp2ir_lus_results.zip lus_results lus_results_centred frame_check && ls -la selfp2ir_lus_results.zip